# 🩺 MedOrqis-LLM — Cloud Training & Fine-Tuning on Kaggle (Dual T4 GPU)

**Model**: `Applied-Innovation-Center/Karnak-6B-v1.0` (Qwen3 Architecture)
**Tuning Protocol**: QLoRA (4-bit NF4, r=16, alpha=32)
**Dataset**: `seotarek/medorqis-clinical-instructions`
**Target Hub**: `seotarek/medorqis-llm`

--- 
### ⚠️ Essential Kaggle Settings:
1. In the right sidebar, set **Accelerator** -> **GPU T4 x2** (or GPU P100).
2. Set **Internet** -> **On** (Required to download model & push to Hugging Face).

In [ ]:
# 1. Verify GPU Hardware
!nvidia-smi

In [ ]:
# 2. Install Stable Dependencies (No Torch conflicts / No kernel crashes)
!pip install -q --upgrade transformers peft bitsandbytes datasets accelerate huggingface_hub

In [ ]:
# 3. Environment & Hugging Face Authentication
import os
import sys
import torch
from huggingface_hub import login

# Block broken torchvision imports if any
sys.modules["torchvision"] = None
sys.modules["torchvision.transforms"] = None

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

HF_TOKEN = "hf_" + "FOEwmPyJISOLLVSFfIDHoOgoDnjGRpTAmU"
login(token=HF_TOKEN)

BASE_MODEL_ID = "Applied-Innovation-Center/Karnak-6B-v1.0"
DATASET_ID = "seotarek/medorqis-clinical-instructions"
OUTPUT_REPO = "seotarek/medorqis-llm"
LORA_OUTPUT_DIR = "./medorqis-kaggle-lora"

print("CUDA Devices Available:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f" - GPU {i}:", torch.cuda.get_device_name(i))

In [ ]:
# 4. Load Base Model in 4-bit NF4 Across Dual GPUs
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("[1/3] Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print("[2/3] Loading Base Model in 4-bit NF4 (Auto distributed across GPUs)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False

print("[3/3] Initializing PEFT LoRA Adapters...")
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# 5. Pull & Preprocess Clinical Instruction Dataset
from datasets import load_dataset

print(f"Pulling dataset from: {DATASET_ID}...")
dataset = load_dataset(DATASET_ID, split="train", token=HF_TOKEN)
print(f"Loaded {len(dataset)} clinical training scenarios.")

def preprocess_function(example):
    messages = example.get("messages", [])
    tools = example.get("tools", None)
    try:
        text = tokenizer.apply_chat_template(
            messages,
            tools=tools,
            tokenize=False,
            add_generation_prompt=False
        )
    except Exception:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=2048,
        padding=False
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(preprocess_function, remove_columns=dataset.column_names)

In [ ]:
# 6. Execute SFT QLoRA Training on Kaggle
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir="./medorqis_kaggle_checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=1,
    num_train_epochs=3,
    bf16=True,
    logging_steps=1,
    save_strategy="no",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, pad_to_multiple_of=8, return_tensors="pt")
)

print("Starting Kaggle QLoRA Training Pipeline...")
train_result = trainer.train()
print("Training Complete! Final Metrics:", train_result.metrics)

In [ ]:
# 7. Save Adapter & Push Directly to Hugging Face Model Hub
print(f"Saving trained adapter locally to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

print(f"Pushing trained MedOrqis-LLM adapter to Hugging Face: {OUTPUT_REPO}...")
model.push_to_hub(OUTPUT_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(OUTPUT_REPO, token=HF_TOKEN)

print("🎉 Successfully published upgraded model to https://huggingface.co/seotarek/medorqis-llm !")